In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
HUGGINGFACEHUB_ACCESS_TOKEN = userdata.get("HF_TOKEN")

In [ ]:
# !pip install langchain-groq
# !pip install langchain-huggingface
# !pip install -q huggingface_hub
# !pip install grandalf
# !pip install langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.7 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import importlib
import langchain_core.runnables.graph_ascii as ga
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from huggingface_hub import InferenceClient

In [ ]:
model1 = ChatGroq(model="qwen/qwen3.8-27b", api_key=GROQ_API_KEY)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token = HUGGINGFACEHUB_ACCESS_TOKEN,
    task = "text-generation",
    max_new_tokens = 1024, # increase generation length
)
model2 = ChatHuggingFace(llm = llm)

In [ ]:
strparser = StrOutputParser()

## 1. RunnableSequence
* RunnableSequence is a sequential chain of runnables in LangChain that executes each step one after another, passing the output of one step as the input to the next.

* It is useful when you need to compose multiple runnables together in a structured workflow.

In [ ]:
from langchain_classic.schema.runnable import RunnableSequence

In [ ]:
prompt1 = PromptTemplate(
    template='Write a 3 line on following topic. /n {topic}',
    input_variables=['topic']
)

In [ ]:
seq_chain1 = RunnableSequence(prompt1, model2, strparser)

result1 = seq_chain1.invoke({'topic':'Runnables in Langchain'})
print(result1)

Here are three lines on the topic of "Runnables in LangChain":

In LangChain, runnables are a way to execute functions asynchronously, allowing for efficient and concurrent processing of tasks. By utilizing runnables, developers can offload computationally expensive operations and improve the overall performance of their LangChain applications. Runnables provide a flexible and scalable approach to handling tasks, making it easier to build complex and robust AI systems.


In [ ]:
prompt2 = PromptTemplate(
    template = "write one question from following text. \n {text}",
    input_variables = ['text']
)

In [ ]:
seq_chain2 = RunnableSequence(prompt1, model2, strparser, prompt2, model1, strparser)

result2 = seq_chain2.invoke({'topic':'Runnables in Langchain'})
print(result2)

What are the primary benefits of using LangChain's runnables for model development?


In [ ]:
seq_chain2.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
   +-----------------+     
   | ChatHuggingFace |     
   +-----------------+     
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

## 2. RunnableParallel
* RunnableParallel is a runnable primitive that allows multiple runnables to execute in parallel.

* Each runnable receives the same input and processes it independently, producing a dictionary of outputs.

In [ ]:
from langchain_classic.schema.runnable import RunnableParallel

In [ ]:
par_prompt1 = PromptTemplate(
    template='Write only 2 line definition on following topic. /n {topic}',
    input_variables=['topic']
)
par_prompt2 = PromptTemplate(
    template = "write 2 simple questions on following topic. \n {topic}",
    input_variables = ['topic']
)

In [ ]:
par_chain = RunnableParallel({
    'definition': RunnableSequence(par_prompt1, model2, strparser),
    'questions': RunnableSequence(par_prompt2, model1, strparser)
})

par_result = par_chain.invoke({'topic':'Runnables in Langchain'})

Here is a 2-line definition on the topic "Runnables in LangChain":

Runnables in LangChain refer to executable functions that can be triggered at runtime, allowing for dynamic and flexible execution of tasks within the LangChain framework. These runnables can be used to perform various operations, such as executing models, fetching data, or generating responses, making LangChain a highly customizable and adaptable platform.
Here are two simple questions on Runnables in LangChain:

1. What is the primary benefit of using LangChain Runnables for building complex LLM applications?
2. Which two key methods are commonly used to invoke a Runnable and stream its output, respectively?


In [ ]:
print('Definition: ',par_result['definition'])
print('-'*200)
print('\nQuestions: ',par_result['questions'])

Definition:  Here is a 2-line definition on the topic "Runnables in LangChain":

Runnables in LangChain refer to executable functions that can be triggered at runtime, allowing for dynamic and flexible execution of tasks within the LangChain framework. These runnables can be used to perform various operations, such as executing models, fetching data, or generating responses, making LangChain a highly customizable and adaptable platform.
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Questions:  Here are two simple questions on Runnables in LangChain:

1. What is the primary benefit of using LangChain Runnables for building complex LLM applications?
2. Which two key methods are commonly used to invoke a Runnable and stream its output, respectively?


In [ ]:
par_chain.get_graph().print_ascii()

      +-------------------------------------+      
      | Parallel<definition,questions>Input |      
      +-------------------------------------+      
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
+-----------------+               +----------+     
| ChatHuggingFace |               | ChatGroq |     
+-----------------+               +----------+     
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP

## 3. RunnablePassthrough
* RunnablePassthrough is a special Runnable primitive that simply returns the input as output without modifying it.

In [ ]:
from langchain_classic.schema.runnable import RunnablePassthrough

In [ ]:
input_seq = RunnableSequence(prompt1, model2, strparser)
par_seq = RunnableParallel({
    'definition': RunnablePassthrough(),
    'questions': RunnableSequence(prompt2, model1, strparser)
})
pass_chain = RunnableSequence(input_seq, par_seq)

pass_result = pass_chain.invoke({'topic':'Langchain'})

In [ ]:
print('Definition: \n', pass_result['definition'])
print('-'*200)
print('Question: \n', pass_result['questions'])

Definition: 
 It seems like you want me to write a 3-line text on the topic "Langchain". Here it is:

Langchain is a decentralized platform that enables users to create, share, and execute AI models. It utilizes blockchain technology to provide a secure, transparent, and community-driven environment for AI model development and deployment. By leveraging blockchain, Langchain aims to democratize AI model creation and usage, promoting a more equitable and collaborative AI ecosystem.
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Question: 
 How does Langchain utilize blockchain technology to promote a more equitable and collaborative AI ecosystem?


In [ ]:
pass_chain.get_graph().print_ascii()

                 +-------------+                 
                 | PromptInput |                 
                 +-------------+                 
                         *                       
                         *                       
                         *                       
                +----------------+               
                | PromptTemplate |               
                +----------------+               
                         *                       
                         *                       
                         *                       
               +-----------------+               
               | ChatHuggingFace |               
               +-----------------+               
                         *                       
                         *                       
                         *                       
               +-----------------+               
               | StrOutputParser |               


## 4. RunnableLambda
* RunnableLambda is a runnable primitive that allows you to apply custom Python functions within an Al pipeline.

* It acts as a middleware between different Al components, enabling preprocessing, transformation, API calls, filtering, and post-processing in a LangChain workflow.

In [ ]:
from langchain_classic.schema.runnable import RunnableLambda

In [ ]:
input_seq2 = RunnableSequence(par_prompt1, model2, strparser)
par_seq2 = RunnableParallel({
    'definition': RunnablePassthrough(),
    'word_count': RunnableLambda(lambda x:len(x.split()))
})
lam_chain = RunnableSequence(input_seq2, par_seq2)

lam_result = lam_chain.invoke({'topic': 'RunnableLambda in Langchain'})

In [ ]:
print('Definition:\n', lam_result['definition'])
print('-'*200)
print('Word Count:\n', lam_result['word_count'])

Definition:
 Here is a 2-line definition for "Runnable Lambda in LangChain":

A Runnable Lambda in LangChain is a function that can be executed on a LangChain model, allowing for flexible and dynamic interactions with the AI. This enables users to perform custom actions and leverage the model's capabilities in a more programmable way.
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Word Count:
 52


In [ ]:
lam_chain.get_graph().print_ascii()

             +-------------+               
             | PromptInput |               
             +-------------+               
                    *                      
                    *                      
                    *                      
            +----------------+             
            | PromptTemplate |             
            +----------------+             
                    *                      
                    *                      
                    *                      
           +-----------------+             
           | ChatHuggingFace |             
           +-----------------+             
                    *                      
                    *                      
                    *                      
           +-----------------+             
           | StrOutputParser |             
           +-----------------+             
                    *                      
                    *           

## 5. RunnableBranch
* RunnableBranch is a control flow component in LangChain that allows you to conditionally route input data to different chains or runnables based on custom logic.

* It functions like an if/elif/else block for chains - where you define a set of condition functions, each associated with a runnable (e.g., LLM call, prompt chain, or tool). The first matching condition is executed. If no condition matches, a default runnable is used (if provided).

In [ ]:
from langchain_classic.schema.runnable import RunnableBranch
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [ ]:
class Sentiment(BaseModel):
  sentiment: Literal['positive', 'negative'] = Field(description='give the sentiment of review')

pydantic_parser = PydanticOutputParser(pydantic_object=Sentiment)

In [ ]:
br_prompt1 = PromptTemplate(
    template='Write a one line review on the following product. \n {topic}',
    input_variables=['topic']
)
br_prompt2 = PromptTemplate(
    template = 'Classify the sentiment of review text into Positive or Negative. \n {review} \n {format_instruction}',
    input_variables=['review'],
    partial_variables={'format_instruction': pydantic_parser.get_format_instructions()}
)
br_prompt3 = PromptTemplate(
    template = 'Write one line appropriate respose to this positive feedback. \n {feedback}',
    input_variables=['feedback']
)
br_prompt4 = PromptTemplate(
    template = 'Write one line appropriate respose to this negative feedback. \n {feedback}',
    input_variables=['feedback']
)

In [ ]:
review_chain = RunnableSequence(br_prompt1, model2, strparser)
classifier_chain = RunnableParallel({
    'review': RunnablePassthrough(),
    'sentiment': RunnableSequence(br_prompt2, model1, pydantic_parser)
})
branch_chain = RunnableBranch(
    (lambda x: x['sentiment'].sentiment == 'positive',
     (lambda x: {'feedback': x['review']}) | br_prompt3 | model2 | strparser),
    (lambda x: x['sentiment'].sentiment == 'negative',
     (lambda x: {'feedback': x['review']}) | br_prompt4 | model2 | strparser),
    RunnableLambda(lambda x: 'neutral sentiment')
)
final_chain = RunnableSequence(review_chain, classifier_chain, branch_chain)

test_res = final_chain.invoke({'topic': 'Titan Edge Watch'})
print(test_res)

"We're thrilled that you appreciate the perfect blend of style and substance in the Titan Edge Watch, and we're committed to continuing to push the boundaries of innovation and excellence in our products."


In [ ]:
final_chain.get_graph().print_ascii()

                   +-------------+                 
                   | PromptInput |                 
                   +-------------+                 
                           *                       
                           *                       
                           *                       
                  +----------------+               
                  | PromptTemplate |               
                  +----------------+               
                           *                       
                           *                       
                           *                       
                 +-----------------+               
                 | ChatHuggingFace |               
                 +-----------------+               
                           *                       
                           *                       
                           *                       
                 +-----------------+               
            

## 6. LCEL
* RunnableSequence(prompt, model, parser) -> prompt | model | parser
